In [3]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [4]:
import os
os.chdir("D:/Violence_detection")
print(os.getcwd())

D:\Violence_detection


In [5]:
processed_path = r"processed_data"

In [6]:
violence_path = os.path.join(processed_path, "V")
nonviolence_path = os.path.join(processed_path, "NV")

In [7]:
from collections import defaultdict
video_frames = defaultdict(list)

# Violence frames
for frame in os.listdir(violence_path):
    parts = frame.split("_")
    video_id = f"V_{parts[1]}"
    video_frames[video_id].append(os.path.join(violence_path, frame))

# NonViolence frames
for frame in os.listdir(nonviolence_path):
    parts = frame.split("_")
    video_id = f"NV_{parts[1]}"
    video_frames[video_id].append(os.path.join(nonviolence_path, frame))

# Sort frames for LSTM
for vid in video_frames:
    video_frames[vid].sort()

print("Total videos:", len(video_frames))

Total videos: 2350


In [8]:
video_ids = list(video_frames.keys())

print("Example video IDs:", video_ids[:10])
print("Total videos:", len(video_ids))

Example video IDs: ['V_1000', 'V_1001', 'V_1002', 'V_1003', 'V_1004', 'V_1005', 'V_1006', 'V_1007', 'V_1008', 'V_1009']
Total videos: 2350


In [9]:
labels = []

for vid in video_ids:
    if vid.startswith("V"):
        labels.append(1)
    else:
        labels.append(0)

print(labels[:20])
print(labels[-20:])

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [10]:
from sklearn.model_selection import train_test_split

train_ids, test_ids, train_labels, test_labels = train_test_split(
    video_ids,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Train videos:", len(train_ids))
print("Test videos:", len(test_ids))

Train videos: 1880
Test videos: 470


In [11]:
import cv2
import numpy as np
import tensorflow as tf

class VideoGenerator(tf.keras.utils.Sequence):

    def __init__(self, video_ids, labels, video_frames, batch_size=4):
        self.video_ids = video_ids
        self.labels = labels
        self.video_frames = video_frames
        self.batch_size = batch_size

    def __len__(self):
        return int(np.ceil(len(self.video_ids) / self.batch_size))

    def __getitem__(self, idx):

        batch_ids = self.video_ids[idx*self.batch_size:(idx+1)*self.batch_size]
        batch_labels = self.labels[idx*self.batch_size:(idx+1)*self.batch_size]

        X = []
        y = []

        for vid, label in zip(batch_ids, batch_labels):

            frames = sorted(self.video_frames[vid])

            sequence = []

            for frame in frames:
                img = cv2.imread(frame)
                img = cv2.resize(img,(160,160))
                img = img / 255.0
                sequence.append(img)

            X.append(sequence)
            y.append(label)

        return np.array(X), np.array(y)

In [12]:
train_gen = VideoGenerator(train_ids, train_labels, video_frames, batch_size=4)
test_gen = VideoGenerator(test_ids, test_labels, video_frames, batch_size=4)

In [13]:
X, y = train_gen[0]
print(X.shape, y.shape)

(4, 16, 160, 160, 3) (4,)


In [14]:
#MODEL ARCHITECTURE

In [15]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("GPUs available:", gpus)

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [16]:
import sys
import tensorflow as tf
import numpy as np
import cv2

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)

Python: 3.10.9 (tags/v3.10.9:1dd9be6, Dec  6 2022, 20:01:21) [MSC v.1934 64 bit (AMD64)]
TensorFlow: 2.10.0
NumPy: 1.23.5
OpenCV: 4.7.0


In [18]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import TimeDistributed, GlobalAveragePooling2D
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.optimizers import Adam

# Load pretrained MobileNet
mobilenet = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(160,160,3)
)

# Fine-tuning: unfreeze top layers
mobilenet.trainable = True
for layer in mobilenet.layers[:-30]:
    layer.trainable = False

# Build model
model = Sequential()

model.add(TimeDistributed(mobilenet, input_shape=(16,160,160,3)))
model.add(TimeDistributed(GlobalAveragePooling2D()))

model.add(Bidirectional(LSTM(64)))

model.add(Dense(64, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))

model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer=Adam(learning_rate=0.00005),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 time_distributed_2 (TimeDis  (None, 16, 5, 5, 1280)   2257984   
 tributed)                                                       
                                                                 
 time_distributed_3 (TimeDis  (None, 16, 1280)         0         
 tributed)                                                       
                                                                 
 bidirectional (Bidirectiona  (None, 128)              688640    
 l)                                                              
                                                                 
 dense_2 (Dense)             (None, 64)                8256      
                                                                 
 batch_normalization (BatchN  (None, 64)               256       
 ormalization)                                        

In [20]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.10.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [21]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=30,
    callbacks=[early_stop]
)

Epoch 1/30
470/470 [==============================] - 396s 841ms/step - loss: 0.4496 - accuracy: 0.7979 - val_loss: 0.4595 - val_accuracy: 0.8106
Epoch 2/30
470/470 [==============================] - 567s 1s/step - loss: 0.3404 - accuracy: 0.8527 - val_loss: 0.3426 - val_accuracy: 0.8404
Epoch 3/30
470/470 [==============================] - 56s 119ms/step - loss: 0.2654 - accuracy: 0.8888 - val_loss: 0.2873 - val_accuracy: 0.8787
Epoch 4/30
470/470 [==============================] - 84s 178ms/step - loss: 0.2222 - accuracy: 0.9112 - val_loss: 0.2756 - val_accuracy: 0.8830
Epoch 5/30
470/470 [==============================] - 126s 268ms/step - loss: 0.2202 - accuracy: 0.9154 - val_loss: 0.2550 - val_accuracy: 0.8872
Epoch 6/30
470/470 [==============================] - 131s 279ms/step - loss: 0.1861 - accuracy: 0.9314 - val_loss: 0.2686 - val_accuracy: 0.8894
Epoch 7/30
470/470 [==============================] - 344s 731ms/step - loss: 0.1851 - accuracy: 0.9229 - val_loss: 0.2384 - val_

In [22]:
model.save("violence_model_TL.h5")

In [2]:
import os
os.chdir("D:/Violence_detection")
print(os.getcwd())

D:\Violence_detection


In [8]:
import cv2
import numpy as np
import time
from tensorflow.keras.models import load_model

model = load_model("violence_model_TL.h5")

sequence = []
max_frames = 16

cap = cv2.VideoCapture(0)

prev_time = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ---------- FPS CALC ----------
    current_time = time.time()
    fps = 1 / (current_time - prev_time) if prev_time != 0 else 0
    prev_time = current_time

    # ---------- PREPROCESS ----------
    img = cv2.resize(frame, (160,160))
    img = img / 255.0

    sequence.append(img)

    # keep last 16 frames
    if len(sequence) > max_frames:
        sequence.pop(0)

    label = "Loading..."
    color = (255,255,0)

    # ---------- PREDICTION ----------
    if len(sequence) == max_frames:
        input_data = np.expand_dims(sequence, axis=0)

        pred = model.predict(input_data, verbose=0)[0][0]

        if pred > 0.5:
            label = "VIOLENCE"
            color = (0,0,255)
        else:
            label = "NON-VIOLENCE"
            color = (0,255,0)

    # ---------- DISPLAY ----------
    cv2.putText(frame, f"{label}",
                (10,30),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, color, 2)

    cv2.putText(frame, f"FPS: {int(fps)}",
                (10,70),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (255,255,255), 2)

    cv2.imshow("Violence Detection", frame)

    # ---------- EXIT ----------
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [7]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load model
model = load_model("violence_model_TL.h5")

# Video path (CHANGE THIS)
video_path = r"dataset\Real Life Violence Dataset\NonViolence\NV_1119.mp4"

cap = cv2.VideoCapture(video_path)

sequence = []
max_frames = 16

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # preprocess frame
    img = cv2.resize(frame, (160,160))
    img = img / 255.0

    sequence.append(img)

    # keep last 16 frames
    if len(sequence) > max_frames:
        sequence.pop(0)

    label = "Loading..."
    color = (255,255,0)

    # prediction
    if len(sequence) == max_frames:
        input_data = np.expand_dims(sequence, axis=0)

        pred = model.predict(input_data, verbose=0)[0][0]

        if pred > 0.5:
            label = "VIOLENCE"
            color = (0,0,255)
        else:
            label = "NON-VIOLENCE"
            color = (0,255,0)

    # display
    cv2.putText(frame, label,
                (10,30),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, color, 2)

    cv2.imshow("Video Prediction", frame)

    # press q to exit
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()